In [21]:
import pandas as pd
import anndata as ad
import scanpy as sc
from scipy import sparse
import numpy as np

In [11]:
adata_sc =  sc.read_h5ad('../data/all_samples.h5ad')
adata_sc

AnnData object with n_obs × n_vars = 295141 × 21237
    obs: 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'Sample', 'Sample_type', 'Age', 'Origin', 'batch', 'S_score', 'G2M_score', 'phase', 'leiden', 'ann_clusters', 'primate_score', 'background_score'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'ann_clusters_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'

In [13]:
adata_sc.obs['ann_clusters'].value_counts()

ann_clusters
Mesenchyme      47332
Fibroblast      37627
Osteogenic      36809
Dermal          22589
Progenitor      21059
Myoblast        20689
Stroma          19565
Chondrogenic    18186
Neuronal        16248
Meningeal       16055
Subdermal        9400
Perivascular     8537
Glia             6169
Immune           4911
Endothelia       4094
Erythrocyte      3590
Epithelia        2281
Name: count, dtype: int64

In [15]:
target_clusters = adata_sc[~adata_sc.obs['ann_clusters'].isin(['Meningeal', 'Osteogenic', 'Subdermal', 'Epithelia','Erythrocyte', 'Neuronal', 'Endothelia', 'Immune'])]

In [16]:
target_clusters

View of AnnData object with n_obs × n_vars = 201753 × 21237
    obs: 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'Sample', 'Sample_type', 'Age', 'Origin', 'batch', 'S_score', 'G2M_score', 'phase', 'leiden', 'ann_clusters', 'primate_score', 'background_score'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'ann_clusters_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'

In [32]:
X = adata_sc.X

# percent of cells with non-zero expression
pct = np.asarray((X > 0).mean(axis=0)).ravel()

# mean expression
mean_expr = np.asarray(X.mean(axis=0)).ravel()

expr_df = pd.DataFrame({
    'gene': target_clusters.var_names,
    'pct': pct,
    'mean_expr': mean_expr
})

expressed_genes = expr_df[
    (expr_df['pct'] > 0.05) &
    (expr_df['mean_expr'] > 0.1)
]['gene']

In [33]:
expressed_genes

0             A1BG
3              A2M
6        A2ML1-AS1
12            AACS
18           AADAT
           ...    
21228       ZWILCH
21232         ZXDC
21234       ZYG11B
21235          ZYX
21236        ZZEF1
Name: gene, Length: 8331, dtype: object

In [35]:
TF_db = pd.read_csv('../data/5_HomoS_TFs_database.txt', sep='\t', header=0)
TF_set = set(TF_db['Symbol'].to_list())

In [36]:
target_TFs = set(expressed_genes) & TF_set
len(target_TFs)

651

In [42]:
with open('../data/5_targeted_TFs.txt', 'w') as f:
    for i in target_TFs:
        f.write(f'{i}\n')